# Camada Bronze — State of Data Brasil (edição 2024/2025)

Dono: Maycon

Responsabilidade desta etapa: ingerir o CSV bruto do S3 (raw) **sem nenhuma
limpeza de conteúdo**, só aplicando um schema e convertendo para Parquet.
Nulo continua nulo, categoria continua exatamente como veio da pesquisa.
A limpeza de verdade acontece só na Silver.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

from utils.config import get_spark_session
from utils.functions import ler_csv_bruto
from utils.constants import RAW_PATH, BRONZE_PATH, EDICAO

spark = get_spark_session()
print(f"Edição: {EDICAO}")
print(f"Lendo de: {RAW_PATH}")

26/08/07 19:22:32 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/07 19:22:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/07 19:22:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Edição: 2024_2025
Lendo de: ../../data/raw/state_of_data_2024_2025.csv


In [2]:
df_raw = ler_csv_bruto(spark, RAW_PATH)

print("Linhas:", df_raw.count())
print("Colunas:", len(df_raw.columns))

Linhas: 5217
Colunas: 403


## Grava em Parquet (Bronze)

No Glue, essa mesma escrita também dispara a catalogação da tabela `bronze_state_of_data_2024_2025` no Glue Data Catalog (via crawler ou `awswrangler`/`boto3` — a decidir com o grupo qual dos dois usar).

In [3]:
df_raw.write.mode("overwrite").parquet(BRONZE_PATH)
print("Bronze gravada em:", BRONZE_PATH)

26/08/07 19:22:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Bronze gravada em: ../../data/bronze/state_of_data_2024_2025.parquet


## Conferência pós-escrita

Garantir que não perdemos linha/coluna na conversão CSV → Parquet.

In [4]:
df_check = spark.read.parquet(BRONZE_PATH)
assert df_check.count() == df_raw.count(), "Divergência de linhas entre raw e bronze!"
assert len(df_check.columns) == len(df_raw.columns), "Divergência de colunas entre raw e bronze!"
print("OK -- Bronze bate 1:1 com o raw:", df_check.count(), "linhas,", len(df_check.columns), "colunas")

OK -- Bronze bate 1:1 com o raw: 5217 linhas, 403 colunas
